# vLLM Deployment Tests

Run these tests in order. Start by creating and validating an AWS GPU instance, then move through each vLLM deployment phase from the README.

## Phase 0: AWS GPU Instance Setup Test

- Launch an EC2 GPU instance such as `g5.xlarge`.
- Use Ubuntu Deep Learning AMI or Ubuntu 22.04 with NVIDIA drivers.
- Allow SSH from your IP.
- Allow TCP `8000` from your IP for API testing.
- SSH into the instance.
- Confirm the GPU with `nvidia-smi`.

In [4]:
from __future__ import annotations
import functools
import time
import logging

logger = logging.getLogger(__name__)


def track_latency(name: str | None = None):
    def wrapper(func):
        @functools.wraps(func)
        def inner(*args, **kwargs):
            start = time.time()
            try:
                return func(*args, **kwargs)

            finally:
                end = time.time() - start

                logger.info(
                    "%s completed in %.2f ms",
                end
                )

        return inner

    return wrapper

# Phase 0: Download Model

## Phase 1: Basic Local Server Test

Goal: confirm vLLM can start a model server on the default local endpoint.

In [5]:
IPADDRESS = '13.233.117.119'
url = f"http://{IPADDRESS}:8000/v1"

In [ ]:
import requests



headers = {
    "Authorization": "Bearer EMPTY",
    "Content-Type": "application/json",
}

payload = {
    "model": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "messages": [
        {
            "role": "user",
            "content": "Explain transformers in one paragraph."
        }
    ],
    "temperature": 0.7,
    "max_tokens": 256
}

response = requests.post(
    url,
    headers=headers,
    json=payload,
    timeout=60,
)

print(response.status_code)
print(response.json()["choices"][0]["message"]["content"])

ConnectionError: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /v1/chat/completions (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))

## Phase 2: Expose to Network Test

Goal: bind the server to the EC2 network interface so a client can reach port `8000`.

In [ ]:
show_test("Phase 2 command", f"vllm serve {MODEL} --host 0.0.0.0 --port {PORT}")

## Phase 3: Specify GPU Test

Goal: force vLLM to use a selected GPU with `CUDA_VISIBLE_DEVICES`.

In [ ]:
show_test("Phase 3 command", f"CUDA_VISIBLE_DEVICES=0 vllm serve {MODEL}")

## Phases 4-8: Memory and Capacity Tests

Goal: test dtype, tensor parallelism, quantization, max context length, and GPU memory utilization.

In [ ]:
tests = {
    "Phase 4 dtype": f"vllm serve {MODEL} --dtype bfloat16",
    "Phase 5 tensor parallel": f"CUDA_VISIBLE_DEVICES=0,1 vllm serve {MODEL} --tensor-parallel-size 2",
    "Phase 6 quantized model": f"vllm serve {MODEL} --quantization awq",
    "Phase 7 max context": f"vllm serve {MODEL} --max-model-len 8192",
    "Phase 8 GPU memory utilization": f"vllm serve {MODEL} --gpu-memory-utilization 0.90",
}

for name, command in tests.items():
    show_test(name, command)

## Phases 9-13: Throughput and Memory Pressure Tests

Goal: test concurrency, batched tokens, KV cache dtype, CPU offload, and swap space.

In [ ]:
tests = {
    "Phase 9 concurrency": f"vllm serve {MODEL} --max-num-seqs 64",
    "Phase 10 batched tokens": f"vllm serve {MODEL} --max-num-batched-tokens 8192",
    "Phase 11 KV cache dtype": f"vllm serve {MODEL} --kv-cache-dtype fp8",
    "Phase 12 CPU offload": f"vllm serve {MODEL} --cpu-offload-gb 20",
    "Phase 13 swap space": f"vllm serve {MODEL} --swap-space 16",
}

for name, command in tests.items():
    show_test(name, command)

## Phases 14-18: Serving Feature Tests

Goal: test prefix caching, chunked prefill, trust remote code, API key protection, and local model loading.

In [ ]:
tests = {
    "Phase 14 prefix caching": f"vllm serve {MODEL} --enable-prefix-caching",
    "Phase 15 chunked prefill": f"vllm serve {MODEL} --enable-chunked-prefill",
    "Phase 16 trust remote code": f"vllm serve {MODEL} --trust-remote-code",
    "Phase 17 API key": f"vllm serve {MODEL} --api-key {API_KEY}",
    "Phase 18 local model": f"vllm serve {LOCAL_MODEL}",
}

for name, command in tests.items():
    show_test(name, command)

## Phases 19-20: Production Command Tests

Goal: validate full single-GPU and multi-GPU startup commands before production use.

In [ ]:
single_gpu_command = f"""CUDA_VISIBLE_DEVICES=0 vllm serve {LOCAL_MODEL} \\
  --host 0.0.0.0 \\
  --port {PORT} \\
  --dtype bfloat16 \\
  --gpu-memory-utilization 0.95 \\
  --max-model-len 8192 \\
  --max-num-seqs 64 \\
  --max-num-batched-tokens 8192 \\
  --enable-prefix-caching \\
  --enable-chunked-prefill \\
  --api-key {API_KEY}"""

multi_gpu_command = f"""CUDA_VISIBLE_DEVICES=0,1,2,3 vllm serve /models/Llama-3.1-70B-Instruct \\
  --host 0.0.0.0 \\
  --port {PORT} \\
  --tensor-parallel-size 4 \\
  --dtype bfloat16 \\
  --gpu-memory-utilization 0.95 \\
  --max-model-len 8192 \\
  --enable-prefix-caching \\
  --enable-chunked-prefill \\
  --api-key {API_KEY}"""

show_test("Phase 19 single-GPU production", single_gpu_command)
show_test("Phase 20 multi-GPU production", multi_gpu_command)

## API Smoke Test

After a server is running, use this request to confirm the OpenAI-compatible endpoint responds.

In [ ]:
show_test(
    "OpenAI-compatible smoke test",
    f"curl http://localhost:{PORT}/v1/models",
)